# Data Processing Pipeline

This notebook applies a processing pipeline to raw borehole data (Conductivity vs. Depth). Each step of the pipeline is executed sequentially, and a comparative plot is generated to visualize the effect of the transformation.

The processing functions are imported from the `processing.py` script located in the `../modules/` directory.

In [ ]:
# === 1. Setup and Imports ===
# This cell imports necessary libraries and the custom processing functions.

import sys
import os
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import json

# -- Setup project root path to allow imports from other directories --
try:
    root = os.path.abspath('..')
    if root not in sys.path:
        sys.path.append(root)
except NameError:
    # If running in an environment where '..' is not defined, use current dir
    root = os.getcwd()

print(f"Project root set to: {root}")

# -- Import custom processing functions from the module --
try:
    import importlib.util
    # The path to the module is relative to this notebook's location (notebooks/)
    spec = importlib.util.spec_from_file_location("processing_module", "../modules/01_processing.py")
    processing_module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(processing_module)

    # Assign functions to variables for easier access
    process_borehole_data = processing_module.process_borehole_data
    DEFAULT_COLUMN_MAPPINGS = processing_module.DEFAULT_COLUMN_MAPPINGS
    adjust_vertical_position = processing_module.adjust_vertical_position
    filter_non_negative_values = processing_module.filter_non_negative_values
    find_column_name = processing_module.find_column_name
    average_grouped_by_depth = processing_module.average_grouped_by_depth
    resample_profile_uniform = processing_module.resample_profile_uniform
    apply_savgol_filter_to_df = processing_module.apply_savgol_filter_to_df
    
    print("Successfully imported processing functions from '01_processing.py'.")

except (ImportError, FileNotFoundError) as e:
    print(f"Error importing processing module: {e}")
    print("Please ensure 'modules/01_processing.py' exists and is accessible from the project root.")

# Experiment Configuration (Edit Me)

This section centralizes **all important parameters** used throughout the notebook so you can configure a run in one place.

> **How to use:** Adjust the values in the code cell below, then re-run the notebook from top to bottom.


In [ ]:
# --- Centralized Parameters ---
# Adjust these constants to control I/O, processing, and visualization.
# === I/O paths and project metadata ===
# FILE_NAME: Name of the input CSV file containing raw profile measurements (e.g., depth vs conductivity).
FILE_NAME = 'AW6_D_YSI_20250226.csv'
# INPUT_DIR: Folder (relative to project root) where input files are located.
INPUT_DIR = os.path.join(root, 'data', 'raw', 'raw_new')
# FILE_PATH: Full path to the input file. If you set this explicitly, it takes precedence over INPUT_DIR/FILE_NAME.
FILE_PATH = os.path.join(INPUT_DIR, FILE_NAME)
# CSV_ENCODING: Text encoding for reading the input file (e.g., 'utf-8', 'latin-1').
CSV_ENCODING = 'utf-8'
# COLUMN_MAPPINGS: Column aliases used for auto-detection (keys: 'depth', 'conductivity').
# This overrides module defaults (DEFAULT_COLUMN_MAPPINGS) to support custom headers.
COLUMN_MAPPINGS = DEFAULT_COLUMN_MAPPINGS  # or provide your own dict
# === Vertical datum correction ===
# VERTICAL_ADJUST_METHOD: Method to correct vertical position ('TOM' or 'YSI').
VERTICAL_ADJUST_METHOD = 'TOM'
# VERTICAL_ADJUSTMENT_M: Static offset in meters to align the depth reference (positive lowers the profile).
VERTICAL_ADJUSTMENT_M = 0.272
# === Resampling to uniform depth grid ===
# RESAMPLE_DZ_METHOD: Strategy to determine uniform depth increment from raw data ('percentile95','median','mean','min').
RESAMPLE_DZ_METHOD = 'median'
# === Savitzky–Golay smoothing ===
# SAVGOL_WINDOW_LENGTH: Odd integer window length (number of samples) for smoothing.
SAVGOL_WINDOW_LENGTH = 15
# SAVGOL_POLY_ORDER: Polynomial order (0–5 typical) for the Savitzky–Golay filter.
SAVGOL_POLY_ORDER = 3
# === Plotting aesthetics ===
# OPACITY: Trace opacity for the 'before/after' overlays [0–1].
OPACITY = 0.8
# LEGEND_XANCHOR / LEGEND_YANCHOR: Legend anchoring inside the figure ('left'/'right', 'top'/'bottom').
LEGEND_XANCHOR = 'left'
LEGEND_YANCHOR = 'top'
# FIG_HEIGHT: Figure height in pixels.
FIG_HEIGHT = 500

## Plotting Helper Function

To keep the code clean and avoid repetition, we define a helper function to create the comparison plots.

In [ ]:
def plot_comparison(df_before, df_after, name_before, name_after, depth_col, cond_col, title):
    """
    Generates an aesthetic Plotly graph to compare data before and after a processing step.
    
    Args:
        df_before (pd.DataFrame): DataFrame before the processing step.
        df_after (pd.DataFrame): DataFrame after the processing step.
        name_before (str): Label for the 'before' trace.
        name_after (str): Label for the 'after' trace.
        depth_col (str): Name of the depth column.
        cond_col (str): Name of the conductivity column.
        title (str): Title of the plot.
    """
    # Parameterized plotting controls
    _OPACITY = OPACITY if 'OPACITY' in globals() else 0.8
    _LEG_X = LEGEND_XANCHOR if 'LEGEND_XANCHOR' in globals() else 'left'
    _LEG_Y = LEGEND_YANCHOR if 'LEGEND_YANCHOR' in globals() else 'top'
    _FIG_HEIGHT = FIG_HEIGHT if 'FIG_HEIGHT' in globals() else 500
    fig = go.Figure()

    # Add trace for data BEFORE processing
    fig.add_trace(go.Scatter(
        x=df_before[depth_col],
        y=df_before[cond_col],
        mode='markers',
        name=f"{name_before} (n = {len(df_before)})",
        marker=dict(color='royalblue', size=4),
        opacity=_OPACITY
    ))

    # Add trace for data AFTER processing
    fig.add_trace(go.Scatter(
        x=df_after[depth_col],
        y=df_after[cond_col],
        mode='markers',
        name=f"{name_after} (n = {len(df_after)})",
        marker=dict(color='firebrick', size=4)
    ))

    # Update layout for a clean, professional look
    fig.update_layout(
        title=dict(text=f'<b>{title}</b>', x=0.5, font=dict(size=20)),
        xaxis_title=f"<b>{depth_col}</b>",
        yaxis_title=f"<b>{cond_col}</b>",
        xaxis=dict(gridcolor='lightgrey'),
        yaxis=dict(gridcolor='lightgrey'),
        template='plotly_white',
        legend=dict(
            x=0.01,
            y=0.99,
            xanchor=_LEG_X,
            yanchor=_LEG_Y,
            bgcolor='rgba(255, 255, 255, 0.7)',
            bordercolor='black',
            borderwidth=1
        ),
        height=_FIG_HEIGHT
    )
    
    fig.show()

## Step 0: Load Raw Data

First, we load the raw CSV file into a pandas DataFrame. We'll use a sample file name, but you can change `file_name` to process a different file. We also auto-detect the correct column names for depth and conductivity using the mappings from our module.

In [ ]:
# Map configuration constants to legacy variables used below
file_name = FILE_NAME
file_path = FILE_PATH
# === Define file and load data ===
file_name = 'AW6_D_YSI_20250226.csv' # <-- CHANGE THIS TO YOUR FILENAME
file_path = os.path.join(root, 'data', 'raw', 'raw_new', file_name)

try:
    df_raw = pd.read_csv(file_path, encoding=(CSV_ENCODING if 'CSV_ENCODING' in globals() else 'utf-8'))
    print(f"Successfully loaded '{file_name}' with {len(df_raw)} records.")
    display(df_raw.head())
except FileNotFoundError:
    print(f"Error: File not found at '{file_path}'")
    print("Please ensure the file exists and the path is correct.")
    # Create a dummy dataframe to allow the rest of the notebook to run without errors
    df_raw = pd.DataFrame({
        'Vertical Position [m]': [ -1, 0, 1, 2, 2, 3, 4, 5, 6, 7, 8, 9, 10],
        'Corrected sp Cond [uS/cm]': [100, 200, 500, 1000, 1100, 1500, 2500, 4000, 6000, 8000, 10000, 12000, 15000]
    })
    print("\nCreated a dummy DataFrame to demonstrate the pipeline.")

# -- Auto-detect column names --
depth_col = find_column_name(df_raw, 'depth', COLUMN_MAPPINGS)
cond_col = find_column_name(df_raw, 'conductivity', COLUMN_MAPPINGS)

if depth_col and cond_col:
    print(f"\nDetected Depth Column: '{depth_col}'")
    print(f"Detected Conductivity Column: '{cond_col}'")
else:
    print("\nError: Could not detect depth and/or conductivity columns. Please check column names in the file.")

## Step 1: Adjust Vertical Position

The first processing step is to apply a vertical adjustment to the depth measurements. We use the `adjust_vertical_position` function. Here, we'll use the `'TOM'` method as an example.

In [ ]:
if 'df_raw' in locals() and depth_col and cond_col:
    # Apply the adjustment
    df_adjusted = adjust_vertical_position(
        df_raw,
        depth_col=depth_col,
        method=(VERTICAL_ADJUST_METHOD if 'VERTICAL_ADJUST_METHOD' in globals() else 'TOM'), # or 'YSI'
        adjustment=(VERTICAL_ADJUSTMENT_M if 'VERTICAL_ADJUSTMENT_M' in globals() else 0.272)
    )

    # Plot the comparison
    plot_comparison(
        df_before=df_raw,
        df_after=df_adjusted,
        name_before='Raw Data',
        name_after='Adjusted Data',
        depth_col=depth_col,
        cond_col=cond_col,
        title=f'Step 1: Vertical Position Adjustment ({file_name})'
    )

## Step 2: Filter Negative Values

Next, we remove any records with negative depth values, which are physically meaningless for this type of measurement. We apply this to the already adjusted data.

In [ ]:
if 'df_adjusted' in locals() and depth_col and cond_col:
    # Apply the filter
    df_positive = filter_non_negative_values(
        df_adjusted, 
        depth_col=depth_col
    )

    # Plot the comparison
    plot_comparison(
        df_before=df_adjusted, 
        df_after=df_positive,
        name_before='Adjusted Data',
        name_after='Positive Depth Data',
        depth_col=depth_col,
        cond_col=cond_col,
        title=f'Step 2: Removing Negative Depth Values ({file_name})'
    )

## Step 3: Average Duplicates by Depth

Some logging tools may record multiple conductivity values at the exact same depth. This step groups the data by each unique depth value and calculates the mean conductivity, ensuring one value per depth.

In [ ]:
if 'df_positive' in locals() and depth_col and cond_col:
    # Apply the averaging function
    df_averaged, duplicates = average_grouped_by_depth(
        df_positive,
        depth_col=depth_col,
        value_col=cond_col
    )
    
    print(f"Found and averaged {len(duplicates)} depth points with duplicate records.")
    if not duplicates.empty:
        display(duplicates.head())

    # Plot the comparison
    plot_comparison(
        df_before=df_positive,
        df_after=df_averaged,
        name_before='Positive Depth Data',
        name_after='Averaged Data',
        depth_col=depth_col,
        cond_col=cond_col,
        title=f'Step 3: Averaging Duplicate Depths ({file_name})'
    )

## Step 4: Resample to Uniform Grid

The data is often recorded at irregular depth intervals. To standardize the profile, we resample it to a uniform grid using PCHIP interpolation. The spacing `dz` can be set manually or calculated automatically.

In [ ]:
if 'df_averaged' in locals() and depth_col and cond_col:
    # Apply resampling
    df_resampled = resample_profile_uniform(
        df_averaged, 
        depth_col=depth_col, 
        value_col=cond_col, 
        dz_method=(RESAMPLE_DZ_METHOD if 'RESAMPLE_DZ_METHOD' in globals() else 'median') # Options: 'percentile95', 'median', 'mean', 'min'
    )

    # Plot the comparison
    plot_comparison(
        df_before=df_averaged, 
        df_after=df_resampled,
        name_before='Averaged Data',
        name_after='Resampled Data',
        depth_col=depth_col,
        cond_col=cond_col,
        title=f'Step 4: Resampling to Uniform Grid ({file_name})'
    )

## Step 5 (Optional): Smoothing with Savitzky-Golay Filter

Finally, we can apply a Savitzky-Golay filter to smooth out high-frequency noise in the resampled data, revealing the underlying trend more clearly.

In [ ]:
if 'df_resampled' in locals() and depth_col and cond_col:
    # Apply the filter
    df_smoothed = apply_savgol_filter_to_df(
        df_resampled,
        value_col=cond_col,
        window_length=(SAVGOL_WINDOW_LENGTH if 'SAVGOL_WINDOW_LENGTH' in globals() else 15), # Must be odd, will be adjusted if even
        poly_order=(SAVGOL_POLY_ORDER if 'SAVGOL_POLY_ORDER' in globals() else 3)
    )

    # Plot the comparison
    plot_comparison(
        df_before=df_resampled,
        df_after=df_smoothed,
        name_before='Resampled Data',
        name_after='Smoothed Data',
        depth_col=depth_col,
        cond_col=cond_col,
        title=f'Step 5: Savitzky-Golay Smoothing ({file_name})'
    )
    
    print("\nFinal Processed Data Head:")
    display(df_smoothed.head())